# Open This Notebook in Colab

<a href="https://colab.research.google.com/github/dreameraiquest/IncidentIQ/blob/main/notebooks/Multi_Agent_DevOps_Incident_Analysis_Suite_Colab.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Multi-Agent DevOps Incident Analysis Suite — Production-Grade Colab Notebook

This notebook is designed for your hackathon project.

It lets you:

1. Upload raw `.jsonl`, `.log`, `.txt`, or `.zip` log files.
2. Parse and normalize mixed production logs.
3. Detect incident signals across:
   - Database
   - Network
   - Authentication
   - Memory/CPU
   - Deployment regression
   - API timeout
   - Queue backlog
   - Disk/storage
   - External dependency
   - Unknown
4. Run a multi-agent style pipeline:
   - Parser Agent
   - Signal Extraction Agent
   - Classifier Agent
   - Timeline Agent
   - Root Cause Agent
   - Remediation Agent
   - Notification Agent
   - Ticket Agent
   - Cookbook Agent
5. Generate:
   - Incident report
   - Slack-style message preview
   - JIRA/GitHub-style ticket preview
   - Remediation cookbook
   - Eval-ready structured output
6. Optionally score results using `ground_truth_eval_only/`.

> Recommended usage: upload `block1_quick_raw_mixed_logs_v2.zip` first. After your app is stable, upload `block2_full_raw_mixed_logs_v2.zip`.

## 0. Runtime Notes

This notebook works in two modes:

### Mode A — No API key
Uses deterministic signal scoring and heuristic RCA. This is good for the first hackathon demo and offline testing.

### Mode B — LLM-enhanced
You can add OpenAI/Azure/OpenRouter later for better RCA, summarization, and remediation text. The notebook has placeholders for that upgrade.

The key idea: your agents should **not** read labels from logs. They should infer incident type and severity from raw operational signals.

In [1]:
# ============================================================
# 1. Install lightweight dependencies
# ============================================================
# This cell is responsible for installing the necessary Python libraries
# that the notebook will use. It uses `pip` to install packages.
#
# `-q` flag: Stands for 'quiet', meaning `pip` will only output errors
#            and warnings, making the output less verbose.
#
# Libraries installed:
# - `pandas`: A powerful data manipulation and analysis library.
# - `numpy`: A fundamental package for numerical computing with Python.
# - `scikit-learn`: A machine learning library providing various
#                   classification, regression, and clustering algorithms.
# - `tqdm`: A library to display progress bars for loops and iterations.
# - `pydantic`: A data validation and settings management library,
#               used here for defining data models (schemas).
# - `matplotlib`: A comprehensive library for creating static,
#                 animated, and interactive visualizations in Python.
#
# These libraries are crucial for data processing, signal extraction,
# incident classification, and report generation within this notebook.
!pip -q install pandas numpy scikit-learn tqdm pydantic matplotlib

In [31]:
# ============================================================
# 2. Imports and configuration
# ============================================================
# This cell imports all necessary Python libraries and sets up
# global configurations for the notebook.

# Standard library imports:
import os             # For interacting with the operating system (e.g., file paths)
import re             # For regular expression operations (pattern matching)
import json           # For working with JSON data
import zipfile        # For working with ZIP archives
import shutil         # For high-level file operations (e.g., copying files)
import math           # For mathematical operations
import statistics     # For statistical calculations
from pathlib import Path  # For object-oriented filesystem paths
from datetime import datetime, timezone # For working with dates and times, including timezone info
from collections import Counter, defaultdict # For specialized container datatypes

# Third-party library imports:
import numpy as np    # Numerical computing library
import pandas as pd   # Data analysis and manipulation library
from tqdm.auto import tqdm # Progress bar for loops
from pydantic import BaseModel, Field # Data validation and settings management
import matplotlib.pyplot as plt # Plotting library

# Pandas display option:
pd.set_option("display.max_colwidth", 160) # Increases the maximum column width for DataFrame display

# Define working directories as Path objects for easier manipulation:
WORK_DIR = Path("/content/devops_incident_suite") # Base directory for the project
UPLOAD_DIR = WORK_DIR / "uploaded"              # Directory for uploaded log files
EXTRACT_DIR = WORK_DIR / "extracted"            # Directory where ZIPs will be extracted
OUTPUT_DIR = WORK_DIR / "outputs"               # Directory for generated reports and outputs

# Create the working directories if they do not already exist.
# `parents=True` creates any necessary parent directories.
# `exist_ok=True` prevents an error if the directory already exists.
for d in [UPLOAD_DIR, EXTRACT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Print the base working directory to confirm setup.
print("Workspace:", WORK_DIR)

Workspace: /content/devops_incident_suite


## 1. Upload Your Logs

Upload one of these:

- `block1_quick_raw_mixed_logs_v2.zip`
- `block2_full_raw_mixed_logs_v2.zip`
- A folder zipped with raw `.jsonl`, `.log`, or `.txt` files
- Individual `.jsonl`, `.log`, or `.txt` files

The notebook will scan for logs recursively.

In [5]:
# ============================================================
# 3. Upload files
# ============================================================
# This cell allows you to upload log files from your local machine
# directly into the Colab environment. It uses the `google.colab.files`
# utility for this purpose.
#
# `files.upload()`:
#   - This function opens a file selection dialog in your browser,
#     allowing you to choose one or more files to upload.
#   - It returns a dictionary where keys are filenames and values are
#     the file contents as bytes.
#
# Processed files:
#   - Each uploaded file's content is written to a corresponding file
#     within the `UPLOAD_DIR` (defined in the configuration cell).
#   - This ensures that all uploaded logs are stored in a consistent
#     location for subsequent processing by the notebook.
#   - A message is printed for each uploaded file, confirming its path.
from google.colab import files

uploaded = files.upload()

for filename, content in uploaded.items():
    target = UPLOAD_DIR / filename
    with open(target, "wb") as f:
        f.write(content)
    print("Uploaded:", target)

Saving devops_raw_log_corpus_v2_all.zip to devops_raw_log_corpus_v2_all.zip
Uploaded: /content/devops_incident_suite/uploaded/devops_raw_log_corpus_v2_all.zip


In [8]:
import os
import re
import json
import zipfile
import shutil
import math
import statistics
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pydantic import BaseModel, Field
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 160)

WORK_DIR = Path("/content/devops_incident_suite")
UPLOAD_DIR = WORK_DIR / "uploaded"
EXTRACT_DIR = WORK_DIR / "extracted"
OUTPUT_DIR = WORK_DIR / "outputs"

# ============================================================
# 4. Extract ZIPs and discover log files
# ============================================================
# This cell handles the extraction of uploaded ZIP files and the
# discovery of relevant log files within the extracted directories.

def reset_extract_dir():
    """Removes and recreates the extraction directory to ensure a clean state."""
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR) # Recursively delete the directory and its contents
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True) # Create the directory, including any necessary parent directories

def extract_uploads():
    """Extracts uploaded ZIP files and copies other files to the extraction directory."""
    reset_extract_dir() # Start with a clean extraction directory
    for p in UPLOAD_DIR.iterdir(): # Iterate through all items in the upload directory
        if p.is_dir(): # Check if the current item is a directory (e.g., .ipynb_checkpoints)
            continue # Skip directories to prevent errors with shutil.copy2
        if p.suffix.lower() == ".zip": # If the file is a ZIP archive
            print("Extracting:", p.name)
            with zipfile.ZipFile(p, "r") as z:
                # Extract all contents into a new subdirectory named after the zip file (without .zip extension)
                z.extractall(EXTRACT_DIR / p.stem)
        else:
            # If it's not a ZIP, copy the file directly to the extraction directory
            shutil.copy2(p, EXTRACT_DIR / p.name)

def discover_logs(root: Path):
    """Recursively discovers log files within a given root directory."""
    exts = {".jsonl", ".log", ".txt"} # Define accepted log file extensions
    files = [] # Initialize a list to store discovered log files
    for p in root.rglob("*"): # Recursively glob for all files and directories under the root
        if p.is_file() and p.suffix.lower() in exts: # Check if it's a file and has a recognized log extension
            # Avoid scoring labels as runtime inputs. This helps prevent mixing ground truth with raw logs.
            if "ground_truth_eval_only" in str(p):
                continue
            files.append(p) # Add the discovered log file to the list
    return sorted(files) # Return the sorted list of discovered log files

extract_uploads() # Execute the function to extract uploaded files
log_files = discover_logs(EXTRACT_DIR) # Discover log files in the extracted directory

print(f"Discovered {len(log_files)} raw log file(s):")
for p in log_files[:30]: # Print the first 30 discovered log files
    print(" -", p.relative_to(EXTRACT_DIR))
if len(log_files) > 30:
    print(" ...") # Indicate if there are more than 30 files

Extracting: devops_raw_log_corpus_v2_all.zip
Discovered 14 raw log file(s):
 - devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl
 - devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_02.jsonl
 - devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_03.jsonl
 - devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_04.jsonl
 - devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_01.jsonl
 - devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_02.jsonl
 - devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_03.jsonl
 - devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_04.jsonl
 - devops_raw_log_corpus_v2_all/devops_raw_l

## 2. Data Models

These schemas make the pipeline more production-like.

In the real app, these become Pydantic contracts between LangGraph nodes.

In [9]:
# ============================================================
# 5. Pydantic models
# ============================================================
class ParsedLog(BaseModel):
    source_file: str
    line_no: int
    ts: str | None = None
    level: str | None = None
    service: str | None = None
    host: str | None = None
    pod: str | None = None
    region: str | None = None
    env: str | None = None
    version: str | None = None
    trace_id: str | None = None
    span_id: str | None = None
    logger: str | None = None
    message: str
    duration_ms: float | None = None
    endpoint: str | None = None
    status: int | None = None
    raw: dict | str

class IncidentCandidate(BaseModel):
    incident_id: str
    category: str
    severity: str
    confidence: float
    affected_services: list[str]
    evidence_count: int
    evidence_samples: list[dict]
    first_seen: str | None = None
    last_seen: str | None = None
    root_cause: str
    remediation: list[str]
    validation_checks: list[str]
    ticket_required: bool
    notification_required: bool

## 3. Parser Agent

The parser tries JSON first, then falls back to regex/plain-text parsing.

For production hardening:
- Keep raw line and parse errors.
- Never drop malformed lines silently.
- Preserve source file and line number for evidence traceability.

In [10]:
# ============================================================
# 6. Parser Agent
# ============================================================
# This agent is responsible for parsing raw log lines, attempting to extract
# structured information. It first tries to parse lines as JSON, and if that
# fails, it falls back to using regular expressions to extract key fields.

# Define common timestamp patterns to extract time information from log messages.
TIMESTAMP_PATTERNS = [
    r"(?P<ts>\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?Z?)", # ISO 8601 format with optional milliseconds and Z for UTC
    r"(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}(?:,\d+)?)"  # YYYY-MM-DD HH:MM:SS with optional milliseconds
]

# Define a regular expression pattern to extract log levels (e.g., TRACE, INFO, ERROR).
LEVEL_PATTERN = r"\b(?P<level>TRACE|DEBUG|INFO|WARN|WARNING|ERROR|FATAL|CRITICAL)\b"

def normalize_level(level):
    """Normalizes log levels to a consistent set of values."""
    if not level:
        return None
    level = str(level).upper() # Convert to uppercase for consistent comparison
    if level == "WARNING":
        return "WARN"
    if level == "CRITICAL":
        return "FATAL"
    return level

def parse_json_line(obj, source_file, line_no):
    """Parses a log entry that is already a JSON object."""
    # Extract message, checking common keys like 'message', 'msg', 'log'
    msg = obj.get("message") or obj.get("msg") or obj.get("log") or json.dumps(obj)
    return ParsedLog(
        source_file=source_file, # Original file path for traceability
        line_no=line_no,         # Line number in the original file
        ts=obj.get("ts") or obj.get("timestamp") or obj.get("@timestamp") or obj.get("time"), # Timestamp
        level=normalize_level(obj.get("level") or obj.get("severity")), # Normalized log level
        service=obj.get("service") or obj.get("app") or obj.get("component"), # Service name
        host=obj.get("host") or obj.get("node"), # Host information
        pod=obj.get("pod") or obj.get("container"), # Pod/container name
        region=obj.get("region"), # Cloud region
        env=obj.get("env") or obj.get("environment"), # Environment (e.g., prod, dev)
        version=str(obj.get("version")) if obj.get("version") is not None else None, # Application version
        trace_id=obj.get("trace_id") or obj.get("traceId") or obj.get("trace"), # Distributed tracing ID
        span_id=obj.get("span_id") or obj.get("spanId"), # Distributed tracing span ID
        logger=obj.get("logger"), # Logger name
        message=str(msg), # The main log message
        duration_ms=float(obj.get("duration_ms")) if obj.get("duration_ms") is not None else None, # Request duration
        endpoint=obj.get("endpoint") or obj.get("path") or obj.get("uri"), # API endpoint/URI
        status=int(obj.get("status")) if str(obj.get("status", "")).isdigit() else None, # HTTP status code
        raw=obj # Store the original raw JSON object
    )

def parse_text_line(line, source_file, line_no):
    """Parses a plain text log line using regular expressions."""
    ts = None
    level = None
    # Attempt to extract timestamp using defined patterns
    for pat in TIMESTAMP_PATTERNS:
        m = re.search(pat, line)
        if m:
            ts = m.group("ts")
            break
    # Attempt to extract log level
    m = re.search(LEVEL_PATTERN, line, re.IGNORECASE)
    if m:
        level = normalize_level(m.group("level"))
    service = None
    # Attempt to extract service name
    svc_match = re.search(r"(?:service|app|component)=([A-Za-z0-9_.-]+)", line)
    if svc_match:
        service = svc_match.group(1)
    trace = None
    # Attempt to extract trace ID
    trace_match = re.search(r"(?:trace_id|traceId|trace)=([A-Za-z0-9_.:-]+)", line)
    if trace_match:
        trace = trace_match.group(1)
    return ParsedLog(
        source_file=source_file,
        line_no=line_no,
        ts=ts,
        level=level,
        service=service,
        message=line.strip(), # The cleaned log message
        trace_id=trace,
        raw=line # Store the original raw text line
    )

def parse_file(path: Path):
    """Reads a log file line by line and parses each line."""
    rows = []
    # Get relative path for cleaner output and traceability
    rel = str(path.relative_to(EXTRACT_DIR))
    with open(path, "r", encoding="utf-8", errors="replace") as f: # Open file, replace invalid chars
        for idx, line in enumerate(f, start=1): # Iterate line by line, keeping track of line number
            line = line.strip()
            if not line: # Skip empty lines
                continue
            try:
                obj = json.loads(line) # Attempt to parse as JSON
                if isinstance(obj, dict):
                    rows.append(parse_json_line(obj, rel, idx).model_dump()) # If successful, use JSON parser
                else:
                    rows.append(parse_text_line(line, rel, idx).model_dump()) # Fallback to text parser if JSON is not a dict
            except Exception: # If JSON parsing fails, treat as plain text
                rows.append(parse_text_line(line, rel, idx).model_dump())
    return rows

# Initialize an empty list to store all parsed log entries.
parsed = []
# Iterate through each discovered log file and parse its contents.
# tqdm provides a progress bar for this operation.
for p in tqdm(log_files, desc="Parsing log files"):
    parsed.extend(parse_file(p))

# Convert the list of parsed log entries into a Pandas DataFrame for easier analysis.
logs_df = pd.DataFrame(parsed)
print("Parsed rows:", len(logs_df)) # Display the total number of parsed log rows
display(logs_df.head(5)) # Display the first 5 rows of the parsed DataFrame for inspection

Parsing log files:   0%|          | 0/14 [00:00<?, ?it/s]

Parsed rows: 13063


,source_file,line_no,ts,level,service,host,pod,region,env,version,trace_id,span_id,logger,message,duration_ms,endpoint,status,raw
0,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,1,2026-05-08T09:01:00.000Z,DEBUG,kubelet,ip-10-0-5-31,kubelet,us-east-1,prod,3.0.0-rc1,trc-221ed9c3657d,spn-b058aebf,security.auth,metrics scrape completed retry scheduled but completed successfully,43885.0,/api/orders,201,"{'ts': '2026-05-08T09:01:00.000Z', 'level': 'DEBUG', 'service': 'kubelet', 'host': 'ip-10-0-5-31', 'pod': 'kubelet', 'region': 'us-east-1', 'env': 'prod', '..."
1,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,2,2026-05-08T09:01:04.000Z,INFO,node-agent,ip-10-0-4-19,node-agent,us-west-2,prod,2.8.0,trc-88a39a2361a6,spn-d8a73d8a,app.worker,deployment heartbeat version=2.8.2,12596.0,/api/orders,201,"{'ts': '2026-05-08T09:01:04.000Z', 'level': 'INFO', 'service': 'node-agent', 'host': 'ip-10-0-4-19', 'pod': 'node-agent', 'region': 'us-west-2', 'env': 'pro..."
2,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,3,2026-05-08T09:01:06.000Z,INFO,market-data-consumer,ip-10-0-9-17,market-data-consumer-64f9bdc77d,us-west-2,prod,2.8.1,trc-d9bef05e51de,spn-692b044d,db.pool,scheduled cleanup completed removed=152,2285.0,/api/auth/token,200,"{'ts': '2026-05-08T09:01:06.000Z', 'level': 'INFO', 'service': 'market-data-consumer', 'host': 'ip-10-0-9-17', 'pod': 'market-data-consumer-64f9bdc77d', 're..."
3,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,4,2026-05-08T09:01:06.000Z,INFO,pricing-service,ip-10-0-8-44,pricing-service-64f9bdc77d,ap-south-1,prod,2.8.2,trc-71ed726e3081,spn-30f6d941,app.worker,background task completed duration_ms=550,52565.0,/api/risk/check,200,"{'ts': '2026-05-08T09:01:06.000Z', 'level': 'INFO', 'service': 'pricing-service', 'host': 'ip-10-0-8-44', 'pod': 'pricing-service-64f9bdc77d', 'region': 'ap..."
4,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,5,2026-05-08T09:01:06.000Z,INFO,redis-cache,ip-10-0-4-19,redis-cache-566b8d6d44,ap-south-1,prod,2.8.0,trc-deba9eedb15f,spn-40f63904,security.auth,deployment heartbeat version=2.8.2,11818.0,/api/orders,201,"{'ts': '2026-05-08T09:01:06.000Z', 'level': 'INFO', 'service': 'redis-cache', 'host': 'ip-10-0-4-19', 'pod': 'redis-cache-566b8d6d44', 'region': 'ap-south-1..."


## 4. Signal Extraction Agent

This is where the system starts behaving like an incident analyst.

It maps raw text into operational signals.  
This is **not** the same as final classification; it is evidence extraction.

In [22]:
# ============================================================
# 7. Signal Extraction Agent
# ============================================================
# This agent is responsible for identifying specific operational signals
# within the parsed log messages. It uses a set of predefined rules
# to detect patterns indicative of various incident categories.

# Define SIGNAL_RULES: A dictionary where keys are incident categories
# and values are lists of regular expressions. Each regex is a pattern
# that, if found in a log message, indicates the presence of that signal.
# These patterns are carefully chosen to capture relevant keywords and phrases
# associated with specific types of operational issues.
SIGNAL_RULES = {
    "database": [
        r"connection.*not available|connection.*timed out|jdbc connection|remaining connection slots|too many clients|pg_stat_activity|deadlock|lock wait|slow query|sequential scan|work_mem|fsync|checkpoint",
    ],
    "network": [
        r"connection reset|reset by peer|dns|name resolution|no such host|tls handshake|ssl handshake|certificate verify|upstream connect|tcp retransmits|connection refused",
    ],
    "authentication": [
        r"jwt|token|issuer mismatch|signature|jwk|401|403|rbac|authorization|auth middleware|permission|entitlement",
    ],
    "memory_cpu": [
        r"oomkilled|oom killed|heap|oldgen|gc pause|cpu throttling|cpu saturation|cfs_throttled|memory limit|back-off restarting|resource pressure",
    ],
    "deployment_regression": [
        r"rollout|deployment|canary|new error signature|version 2\\.8\\.1|rollback|feature flag|configmap|configuration reloaded|checksum|new code path",
    ],
    "api_timeout": [
        r"504|gateway timeout|request timed out|upstream timed out|proxy_read_timeout|p95 latency|client timeout|circuit breaker|downstream timeout|read timeout",
    ],
    "queue_backlog": [
        r"consumer lag|records-lag|max.poll.interval|queue depth|backlog|retry topic|dead-letter|poison|rebalance|partition assignment|heartbeat failed",
    ],
    "disk_storage": [
        r"no space left|disk pressure|errno=28|ephemeral storage|evicted|logrotate|filesystem free|disk full|iowait|persistent volume|write latency",
    ],
    "external_dependency": [
        r"provider|vendor|third-party|external|rate limit|429|Retry-After|market data vendor|feed delayed|sequence gap|secondary feed",
    ],
    "unknown": [
        r"unexpected state|unclassified|unknown|missing correlation|trace context.*absent|incomplete trace|manual inspection|insufficient evidence|redacted",
    ],
}

# Define SEVERITY_WEIGHTS: A dictionary mapping log levels to numerical weights.
# These weights are used to quantify the impact or importance of a log entry.
# Higher weights are assigned to more critical log levels (e.g., FATAL, ERROR),
# indicating a more severe signal. 'None' covers cases where the log level
# could not be parsed, assigning a default low weight.
SEVERITY_WEIGHTS = {
    "FATAL": 4.0,
    "ERROR": 3.0,
    "WARN": 1.5,
    "INFO": 0.2,
    "DEBUG": 0.05,
    None: 0.1,
}

def extract_signals(row):
    # Concatenate relevant text fields from the log row into a single string
    # for pattern matching. Convert to lowercase for case-insensitive matching.
    text = f"{row.get('message','')} {row.get('service','')} {row.get('logger','')} {row.get('status','')}".lower()
    hits = []
    # Iterate through each defined incident category and its associated patterns.
    for cat, patterns in SIGNAL_RULES.items():
        for pat in patterns:
            # If any pattern for a category is found in the log text,
            # add the category to the 'hits' list and break to avoid redundant checks
            # for the same category on the current log line.
            if re.search(pat, text, flags=re.IGNORECASE):
                hits.append(cat)
                break
    return hits

# Apply the `extract_signals` function to each row of the `logs_df` DataFrame
# to populate the new 'signals' column with a list of detected categories.
logs_df["signals"] = logs_df.apply(extract_signals, axis=1)

# Create a boolean column 'has_signal' which is True if any signals were detected
# for a given log entry (i.e., the 'signals' list is not empty).
logs_df["has_signal"] = logs_df["signals"].apply(lambda x: len(x) > 0)

# Calculate a 'signal_weight' for each log entry.
# This weight is a product of the log's severity weight (from `SEVERITY_WEIGHTS`)
# and the number of signals detected in that log entry. This amplifies the importance
# of logs that are both severe and exhibit multiple problematic signals.
logs_df["signal_weight"] = logs_df.apply(
    lambda r: SEVERITY_WEIGHTS.get(r.get("level"), 0.1) * max(1, len(r["signals"])),
    axis=1
)

# Print the total number of log rows that contain at least one detected signal.
print("Rows with signals:", int(logs_df["has_signal"].sum()))

# Display the first 10 rows of the DataFrame that have signals,
# showing key columns for inspection of the extracted information.
display(logs_df[logs_df["has_signal"]][["source_file","line_no","ts","level","service","message","signals"]].head(10))

Rows with signals: 6606


,source_file,line_no,ts,level,service,message,signals
1,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,2,2026-05-08T09:01:04.000Z,INFO,node-agent,deployment heartbeat version=2.8.2,[deployment_regression]
4,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,5,2026-05-08T09:01:06.000Z,INFO,redis-cache,deployment heartbeat version=2.8.2,[deployment_regression]
5,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,6,2026-05-08T09:01:12.000Z,INFO,api-gateway,deployment heartbeat version=3.0.0-rc1,[deployment_regression]
11,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,12,2026-05-08T09:01:19.000Z,INFO,kafka-broker,background task completed duration_ms=504,[api_timeout]
13,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,14,2026-05-08T09:01:20.000Z,INFO,risk-engine,feature flag evaluated flag=async-notifications value=true,[deployment_regression]
17,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,18,2026-05-08T09:01:28.000Z,INFO,settlement-worker,feature flag evaluated flag=new-auth-flow value=true,[deployment_regression]
19,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,20,2026-05-08T09:01:33.000Z,ERROR,payment-api,deployment heartbeat version=2.8.0,"[deployment_regression, api_timeout]"
21,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,22,2026-05-08T09:01:34.000Z,WARN,external-monitor,metrics scrape completed,[external_dependency]
22,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,23,2026-05-08T09:01:34.000Z,INFO,external-monitor,feature flag evaluated flag=strict-order-validation value=false,"[deployment_regression, external_dependency]"
25,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,26,2026-05-08T09:01:45.000Z,DEBUG,kafka-broker,deployment heartbeat version=2.8.1,[deployment_regression]


## 5. Classifier + Correlation Agent

This agent groups evidence by file and category.  
In a more advanced LangGraph app, you would group by:
- time window
- trace ID
- service dependency graph
- deployment version
- anomaly windows

In [13]:
# ============================================================
# 8. Classifier + Correlation Agent
# ============================================================
# This agent takes the extracted signals and groups them to identify
# potential incidents. It aggregates evidence, calculates weighted scores,
# and classifies the severity of each incident candidate.

# Filter the DataFrame to include only rows that have at least one signal.
# A copy is made to avoid SettingWithCopyWarning in pandas.
evidence_df = logs_df[logs_df["has_signal"]].copy()

# 'Explode' the 'signals' column. This means if a log entry has multiple
# signals (e.g., ['database', 'network']), it will create a separate row
# for each signal, effectively duplicating the original log entry for each signal it contains.
# This allows grouping by individual signal categories.
exploded = evidence_df.explode("signals").rename(columns={"signals": "category"})

# Define the columns by which to group the exploded evidence.
# Grouping by source file and category helps to consolidate related signals.
group_cols = ["source_file", "category"]

# Aggregate the grouped data to create a summary for each incident candidate.
# - `evidence_count`: Total number of log lines contributing to this incident category.
# - `weighted_score`: Sum of `signal_weight` for all relevant log lines.
# - `error_count`: Number of log lines with 'ERROR' level.
# - `warn_count`: Number of log lines with 'WARN' level.
# - `services`: A sorted list of unique services involved in this incident category.
# - `first_seen`: The earliest timestamp of a log line for this incident category.
# - `last_seen`: The latest timestamp of a log line for this incident category.
summary = (
    exploded
    .groupby(group_cols)
    .agg(
        evidence_count=("message", "count"),
        weighted_score=("signal_weight", "sum"),
        error_count=("level", lambda s: int((s == "ERROR").sum())),
        warn_count=("level", lambda s: int((s == "WARN").sum())),
        services=("service", lambda s: sorted(set([x for x in s.dropna().astype(str) if x]))),
        first_seen=("ts", lambda s: min([x for x in s.dropna().astype(str)] or [None])),
        last_seen=("ts", lambda s: max([x for x in s.dropna().astype(str)] or [None])),
    )
    .reset_index()
)

# Function to classify the severity of an incident (P1, P2, P3) based on
# category, evidence count, weighted score, and error count.
# P1 indicates critical, P2 high, P3 medium.
def classify_severity(category, evidence_count, weighted_score, error_count):
    # Define categories that are inherently more critical (P1 candidates).
    p1_categories = {"database", "memory_cpu", "disk_storage", "deployment_regression"}
    if weighted_score >= 120 or error_count >= 35:
        # High score or many errors often indicate P1 or P2.
        return "P1" if category in p1_categories else "P2"
    if weighted_score >= 35 or error_count >= 8:
        # Moderate score or errors indicate P2.
        return "P2"
    return "P3" # Default to P3 for lower impact.

# Apply the severity classification function to each incident summary row.
summary["severity"] = summary.apply(
    lambda r: classify_severity(r["category"], r["evidence_count"], r["weighted_score"], r["error_count"]),
    axis=1
)

# Calculate a confidence score for each incident.
# This score combines the weighted score and evidence volume to give a holistic confidence.
max_score = max(summary["weighted_score"].max(), 1) # Get max weighted score, avoid division by zero
summary["confidence"] = summary.apply(
    lambda r: round(min(0.98, 0.35 + 0.45*(r["weighted_score"]/max_score) + 0.20*min(1, r["evidence_count"]/50)), 2),
    axis=1
)

# Sort the incident summaries by severity (P1 first) and then by weighted score (highest first).
summary = summary.sort_values(["severity","weighted_score"], ascending=[True, False])

# Display the top 20 incident summaries for review.
display(summary.head(20))

,source_file,category,evidence_count,weighted_score,error_count,warn_count,services,first_seen,last_seen,severity,confidence
43,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_04.jsonl,memory_cpu,355,1021.70,194,130,"[kubelet, market-data-consumer, pricing-service, risk-engine]",2026-05-09T15:12:26.000Z,2026-05-09T16:17:36.000Z,P1,0.94
27,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_01.jsonl,database,279,913.50,158,93,"[order-api, payment-api, postgres-primary, trade-booking-service]",2026-05-09T09:20:37.000Z,2026-05-09T10:15:44.000Z,P1,0.90
46,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_05.jsonl,deployment_regression,357,877.35,160,102,"[api-gateway, auth-service, external-monitor, kafka-broker, kubelet, market-data-consumer, nginx-gateway, node-agent, notification-worker, order-api, paymen...",2026-05-09T17:13:06.000Z,2026-05-09T18:17:52.000Z,P1,0.88
62,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_08.jsonl,disk_storage,283,829.60,156,92,"[kubelet, nginx-gateway, node-agent, order-api, postgres-primary]",2026-05-09T23:02:36.000Z,2026-05-09T23:58:52.000Z,P1,0.86
15,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_03.jsonl,deployment_regression,158,297.50,44,42,"[api-gateway, auth-service, external-monitor, kafka-broker, kubelet, market-data-consumer, nginx-gateway, node-agent, notification-worker, order-api, paymen...",2026-05-08T15:18:16.000Z,2026-05-08T16:00:16.000Z,P1,0.66
2,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_01.jsonl,database,77,265.40,47,24,"[payment-api, postgres-primary]",2026-05-08T09:03:15.000Z,2026-05-08T09:36:48.000Z,P1,0.65
60,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_08.jsonl,database,76,251.40,34,33,"[kubelet, node-agent, order-api, postgres-primary]",2026-05-09T23:03:09.000Z,2026-05-09T23:59:08.000Z,P1,0.65
9,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_02.jsonl,disk_storage,55,189.10,28,21,"[kubelet, node-agent, order-api]",2026-05-08T12:24:46.000Z,2026-05-08T12:57:24.000Z,P1,0.62
11,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_02.jsonl,memory_cpu,72,188.70,36,30,"[kubelet, market-data-consumer]",2026-05-08T12:24:29.000Z,2026-05-08T12:58:04.000Z,P1,0.62
68,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_09.jsonl,external_dependency,477,1187.50,238,155,"[auth-service, external-monitor, kubelet, market-data-consumer, nginx-gateway, node-agent, notification-worker, payment-api, postgres-primary, settlement-wo...",2026-05-10T01:22:08.000Z,2026-05-10T02:31:04.000Z,P2,0.98


## 6. Root Cause + Remediation Agent

This cell maps inferred categories to practical incident response guidance.

Later, you can replace or augment this with an LLM agent that writes richer RCA, while keeping this deterministic mapping as a safety baseline.

In [23]:
# ============================================================
# 9. Root Cause + Remediation Agent
# ============================================================
# This agent takes the classified incident candidates and maps them
# to predefined root causes, remediation steps, and validation checks.
# This provides actionable intelligence for incident response teams.

# RCA_LIBRARY: A dictionary defining incident response guidance for each category.
# Each key represents an incident category (e.g., 'database', 'network').
# The value associated with each category is another dictionary containing:
# - 'root_cause': A string describing the most probable root cause.
# - 'remediation': A list of recommended actions to resolve the incident.
# - 'validation': A list of checks to confirm the remediation was successful.
RCA_LIBRARY = {
    "database": {
        "root_cause": "Likely database contention, connection exhaustion, deadlocks, or slow queries causing downstream failures.",
        "remediation": [
            "Inspect DB connection pool utilization and active sessions.",
            "Check pg_stat_activity or DB lock tables for blockers.",
            "Identify slow queries and recent query plan changes.",
            "Temporarily scale application pods or tune pool size if safe.",
            "Rollback recent DB/query-related changes if correlated."
        ],
        "validation": [
            "DB connection wait time returns to baseline.",
            "5xx and timeout rate decreases.",
            "Active sessions and lock waits normalize."
        ]
    },
    "network": {
        "root_cause": "Likely network path, DNS, TLS, or upstream connectivity instability.",
        "remediation": [
            "Check ingress/load balancer health.",
            "Validate DNS resolution from affected pods.",
            "Inspect network policies and recent infra changes.",
            "Check TLS certificate chain and expiry.",
            "Compare packet loss/retransmits across zones."
        ],
        "validation": [
            "DNS lookup success rate normalizes.",
            "Connection reset and TLS errors stop.",
            "Gateway upstream health checks pass."
        ]
    },
    "authentication": {
        "root_cause": "Likely authentication or authorization configuration issue, token validation mismatch, key rotation, or RBAC regression.",
        "remediation": [
            "Verify issuer, audience, and signing key configuration.",
            "Refresh JWK cache and validate identity provider availability.",
            "Compare auth configuration with previous known-good version.",
            "Check RBAC policy changes and entitlement mappings.",
            "Rollback auth config if failures correlate with reload/deploy."
        ],
        "validation": [
            "401/403 rates return to baseline.",
            "Token validation succeeds for known-good tokens.",
            "RBAC decisions match expected permissions."
        ]
    },
    "memory_cpu": {
        "root_cause": "Likely resource pressure from CPU throttling, memory leak, OOM restarts, or JVM GC pauses.",
        "remediation": [
            "Inspect CPU/memory usage and container limits.",
            "Review heap/GC metrics and recent traffic changes.",
            "Scale replicas or temporarily raise resource limits if safe.",
            "Capture heap/thread dump for suspected leak.",
            "Rollback recent release if resource spike started after deploy."
        ],
        "validation": [
            "Pod restarts stop.",
            "CPU throttling and GC pause metrics normalize.",
            "Latency returns within SLO."
        ]
    },
    "deployment_regression": {
        "root_cause": "Likely regression introduced by deployment, feature flag, or configuration change.",
        "remediation": [
            "Compare error rate before and after rollout.",
            "Identify failing version, feature flag, or config checksum.",
            "Rollback to previous known-good version if customer impact is high.",
            "Disable suspect feature flag.",
            "Run smoke tests after rollback or fix."
        ],
        "validation": [
            "Error signature disappears after rollback/flag disable.",
            "Canary analysis passes.",
            "5xx and latency metrics return to baseline."
        ]
    },
    "api_timeout": {
        "root_cause": "Likely downstream latency or timeout cascade causing API/gateway failures.",
        "remediation": [
            "Identify slow upstream dependency and affected endpoint.",
            "Check downstream health and saturation.",
            "Use circuit breaker/fallback where safe.",
            "Avoid simply increasing timeouts unless root cause is understood.",
            "Scale or restart affected downstream service if needed."
        ],
        "validation": [
            "504 and timeout counts decrease.",
            "p95/p99 latency returns to SLO.",
            "Circuit breaker closes successfully."
        ]
    },
    "queue_backlog": {
        "root_cause": "Likely consumer lag, poison messages, retry storm, or consumer group instability.",
        "remediation": [
            "Check consumer lag by topic/partition/group.",
            "Inspect dead-letter queue and retry topics.",
            "Identify poison payloads or schema mismatches.",
            "Scale consumers if processing is healthy but under-capacity.",
            "Stabilize consumer group membership and heartbeat settings."
        ],
        "validation": [
            "Consumer lag trends down.",
            "DLQ growth stops.",
            "Rebalance frequency returns to normal."
        ]
    },
    "disk_storage": {
        "root_cause": "Likely disk exhaustion, log rotation failure, storage pressure, or high IO wait.",
        "remediation": [
            "Check filesystem usage and largest directories.",
            "Clean or archive old logs safely.",
            "Fix log rotation configuration.",
            "Increase volume size or move workload to healthy node.",
            "Inspect persistent volume latency and storage backend."
        ],
        "validation": [
            "Disk pressure clears.",
            "Write failures stop.",
            "Pods stop being evicted."
        ]
    },
    "external_dependency": {
        "root_cause": "Likely third-party provider degradation, throttling, or external feed delay.",
        "remediation": [
            "Check vendor status page and API response patterns.",
            "Enable fallback provider or degraded mode if available.",
            "Apply exponential backoff and respect Retry-After.",
            "Notify stakeholders of external dependency impact.",
            "Track vendor SLA and open support case if needed."
        ],
        "validation": [
            "Provider 5xx/429 rate decreases.",
            "External latency returns to SLA.",
            "Fallback traffic succeeds."
        ]
    },
    "unknown": {
        "root_cause": "Insufficient evidence for deterministic root cause. Additional telemetry or logs are required.",
        "remediation": [
            "Collect additional logs around the affected trace/time window.",
            "Increase logging for missing correlation IDs.",
            "Check recent deployments, config changes, and infra events.",
            "Ask service owner to inspect domain-specific state transitions.",
            "Create follow-up investigation task rather than applying risky remediation."
        ],
        "validation": [
            "Additional telemetry links symptoms to a clear owner.",
            "Unknown exception rate decreases.",
            "Correlation IDs are present in future logs."
        ]
    }
}

# sample_evidence function:
# Retrieves a limited number of log entries (evidence samples) for a given
# source file and incident category. These samples help in providing context
# and concrete examples of the log patterns that triggered the incident.
def sample_evidence(source_file, category, n=5):
    # Filter the 'exploded' DataFrame to get rows matching the specified source_file and category.
    rows = exploded[(exploded["source_file"] == source_file) & (exploded["category"] == category)]
    # Sort the rows by log level (e.g., ERROR, WARN first) and take the top 'n' samples.
    rows = rows.sort_values(["level"], ascending=True).head(n)
    # Define the columns to keep for the evidence samples.
    keep = ["source_file", "line_no", "ts", "level", "service", "message", "trace_id"]
    # Convert the selected rows and columns into a list of dictionaries, suitable for JSON output.
    return rows[keep].fillna("").to_dict(orient="records")

# Initialize an empty list to store the IncidentCandidate objects.
incidents = []
# Iterate through each incident summary generated by the Classifier + Correlation Agent.
for _, r in summary.iterrows():
    # Retrieve RCA and remediation guidance from RCA_LIBRARY based on the incident category.
    # Fallback to 'unknown' if a specific category is not found.
    lib = RCA_LIBRARY.get(r["category"], RCA_LIBRARY["unknown"])
    # Create an IncidentCandidate object for each incident summary.
    # This object aggregates all relevant information: classification, confidence,
    # affected services, evidence samples, timestamps, and RCA/remediation guidance.
    incidents.append(IncidentCandidate(
        incident_id=f"{r['source_file']}::{r['category']}", # Unique identifier for the incident
        category=r["category"],
        severity=r["severity"],
        confidence=float(r["confidence"]),
        affected_services=r["services"],
        evidence_count=int(r["evidence_count"]),
        evidence_samples=sample_evidence(r["source_file"], r["category"], 5), # Get concrete log samples
        first_seen=r["first_seen"],
        last_seen=r["last_seen"],
        root_cause=lib["root_cause"], # Populate root cause from RCA_LIBRARY
        remediation=lib["remediation"], # Populate remediation steps from RCA_LIBRARY
        validation_checks=lib["validation"], # Populate validation checks from RCA_LIBRARY
        ticket_required=r["severity"] in ["P1", "P2"], # Determine if a ticket is required based on severity
        notification_required=r["severity"] in ["P1", "P2"], # Determine if a notification is required based on severity
    ).model_dump()) # Convert the Pydantic model to a dictionary

# Convert the list of IncidentCandidate objects into a Pandas DataFrame
# for easy manipulation and display.
incidents_df = pd.DataFrame(incidents)
# Display the first 20 incident candidates for review.
display(incidents_df.head(20))

,incident_id,category,severity,confidence,affected_services,evidence_count,evidence_samples,first_seen,last_seen,root_cause,remediation,validation_checks,ticket_required,notification_required
0,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_04.jsonl::memory_cpu,memory_cpu,P1,0.94,"[kubelet, market-data-consumer, pricing-service, risk-engine]",355,"[{'source_file': 'devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_04.jsonl', 'line_no': 632, 'ts': '2026-05...",2026-05-09T15:12:26.000Z,2026-05-09T16:17:36.000Z,"Likely resource pressure from CPU throttling, memory leak, OOM restarts, or JVM GC pauses.","[Inspect CPU/memory usage and container limits., Review heap/GC metrics and recent traffic changes., Scale replicas or temporarily raise resource limits if ...","[Pod restarts stop., CPU throttling and GC pause metrics normalize., Latency returns within SLO.]",True,True
1,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_01.jsonl::database,database,P1,0.90,"[order-api, payment-api, postgres-primary, trade-booking-service]",279,"[{'source_file': 'devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_01.jsonl', 'line_no': 603, 'ts': '2026-05...",2026-05-09T09:20:37.000Z,2026-05-09T10:15:44.000Z,"Likely database contention, connection exhaustion, deadlocks, or slow queries causing downstream failures.","[Inspect DB connection pool utilization and active sessions., Check pg_stat_activity or DB lock tables for blockers., Identify slow queries and recent query...","[DB connection wait time returns to baseline., 5xx and timeout rate decreases., Active sessions and lock waits normalize.]",True,True
2,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_05.jsonl::deployment_regression,deployment_regression,P1,0.88,"[api-gateway, auth-service, external-monitor, kafka-broker, kubelet, market-data-consumer, nginx-gateway, node-agent, notification-worker, order-api, paymen...",357,"[{'source_file': 'devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_05.jsonl', 'line_no': 772, 'ts': '2026-05...",2026-05-09T17:13:06.000Z,2026-05-09T18:17:52.000Z,"Likely regression introduced by deployment, feature flag, or configuration change.","[Compare error rate before and after rollout., Identify failing version, feature flag, or config checksum., Rollback to previous known-good version if custo...","[Error signature disappears after rollback/flag disable., Canary analysis passes., 5xx and latency metrics return to baseline.]",True,True
3,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_08.jsonl::disk_storage,disk_storage,P1,0.86,"[kubelet, nginx-gateway, node-agent, order-api, postgres-primary]",283,"[{'source_file': 'devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_08.jsonl', 'line_no': 568, 'ts': '2026-05...",2026-05-09T23:02:36.000Z,2026-05-09T23:58:52.000Z,"Likely disk exhaustion, log rotation failure, storage pressure, or high IO wait.","[Check filesystem usage and largest directories., Clean or archive old logs safely., Fix log rotation configuration., Increase volume size or move workload ...","[Disk pressure clears., Write failures stop., Pods stop being evicted.]",True,True
4,devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_03.jsonl::deployment_regression,deployment_regression,P1,0.66,"[api-gateway, auth-service, external-monitor, kafka-broker, kubelet, market-data-consumer, nginx-gateway, node-agent, notification-worker, order-api, paymen...",158,"[{'source_file': 'devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/raw_logs/quick_mixed_window_03.jsonl', 'line_no': 376, 'ts': '2026-...",2026-05-08T15:18:16.000Z,2

## 7. Timeline Agent

This creates a time-ordered evidence timeline for top incidents.

For a strong demo, show this timeline before showing RCA.

In [24]:
# ============================================================
# 10. Timeline Agent
# ============================================================
# This agent generates a chronological sequence of log events
# related to a specific incident. This timeline provides a quick
# overview of what happened leading up to and during the incident.

def build_timeline(source_file, category, limit=25):
    """Constructs a timeline of log events for a given incident category and source file."""
    # Filter the 'exploded' DataFrame to include only rows pertinent
    # to the specified source file and incident category.
    rows = exploded[(exploded["source_file"] == source_file) & (exploded["category"] == category)].copy()
    # Sort the filtered rows chronologically by timestamp and then by line number.
    # Limit the number of events to 'limit' for conciseness.
    rows = rows.sort_values(["ts", "line_no"]).head(limit)
    # Select and return key columns as a list of dictionaries, which is suitable for JSON output.
    return rows[["ts","level","service","message","line_no","trace_id"]].fillna("").to_dict(orient="records")

# Identify the top 5 incidents based on severity (P1, P2, P3) and then confidence.
# This ensures that the most critical and confidently identified incidents are highlighted first.
top_incidents = incidents_df.sort_values(["severity","confidence"], ascending=[True, False]).head(5)

# Iterate through each of the top incidents to generate and display its timeline.
for _, inc in top_incidents.iterrows():
    print("\n" + "="*100) # Print a separator for readability.
    # Display incident summary information including category, severity, confidence, and source file.
    print(f"INCIDENT: {inc['category']} | {inc['severity']} | confidence={inc['confidence']} | file={inc['incident_id'].split('::')[0]}")
    # Build a timeline for the current incident, fetching up to 10 relevant log events.
    timeline = build_timeline(inc["incident_id"].split("::")[0], inc["category"], 10)
    # Print each event in the timeline, showing timestamp, level, service, line number, and a truncated message.
    for event in timeline:
        print(f"{event['ts']} [{event['level']}] {event['service']} line={event['line_no']} :: {event['message'][:180]}")


INCIDENT: memory_cpu | P1 | confidence=0.94 | file=devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/raw_logs/full_mixed_window_04.jsonl
2026-05-09T15:12:26.000Z [WARN] pricing-service line=95 :: GC pause took 4212ms, allocation failure sample=21726b latency_ms=42345
2026-05-09T15:12:27.000Z [WARN] kubelet line=96 :: heap usage reached 97 percent before restart
2026-05-09T15:12:34.000Z [ERROR] pricing-service line=100 :: GC pause took 4212ms, allocation failure sample=7b7b8e latency_ms=10182
2026-05-09T15:12:45.000Z [ERROR] pricing-service line=104 :: JVM heap pressure high sample=70edfa latency_ms=57589
2026-05-09T15:13:08.000Z [ERROR] risk-engine line=115 :: autoscaler observed sustained CPU saturation sample=0bee42 latency_ms=8793
2026-05-09T15:13:15.000Z [WARN] risk-engine line=118 :: container_cpu_cfs_throttled_periods_total increased sharply
2026-05-09T15:13:18.000Z [ERROR] pricing-service line=121 :: OldGen usage above threshold
2026-05-09T15:13:29.000Z [INF

## 8. Notification Agent and Ticket Agent

This is demo-safe: it generates previews instead of requiring paid Slack/JIRA.

You can later connect:
- Slack webhook
- Microsoft Teams webhook
- Discord webhook
- JIRA API
- GitHub Issues API
- ServiceNow API

In [25]:
# ============================================================
# 11. Notification + Ticket Preview Agents
# ============================================================
# These agents are responsible for generating previews of incident notifications
# (e.g., Slack messages) and tickets (e.g., Jira/GitHub issues).
# This allows for review and human approval before actual creation or sending.

def slack_message_preview(incident):
    """Generates a Slack-style message preview for a given incident."""
    # Assign an emoji based on incident severity for quick visual identification.
    sev_emoji = {"P1": "🚨", "P2": "⚠️", "P3": "ℹ️"}.get(incident["severity"], "ℹ️")
    # Format affected services, showing up to the first 6 for brevity.
    services = ", ".join(incident["affected_services"][:6])
    # Get the message from the first evidence sample, or a default message if no samples exist.
    evidence = incident["evidence_samples"][0]["message"] if incident["evidence_samples"] else "No evidence sample"
    # Construct the Slack message using f-strings for easy formatting.
    return f"""{sev_emoji} *Incident Detected: {incident['category'].replace('_',' ').title()}*

*Severity:* {incident['severity']}
*Confidence:* {incident['confidence']}
*Affected services:* {services}
*Evidence count:* {incident['evidence_count']}
*Likely cause:* {incident['root_cause']}
*Top evidence:* `{evidence[:250]}`
*Recommended first action:* {incident['remediation'][0]}

_Status: Preview only. Human approval required before sending._
"""

def ticket_payload_preview(incident):
    """Generates a dictionary representing a ticket payload for a given incident."""
    # Structure the ticket payload with relevant incident details.
    return {
        "title": f"{incident['severity']}: {incident['category'].replace('_',' ').title()} incident detected",
        "labels": ["ai-generated", "incident", incident["severity"], incident["category"]], # Labels for filtering/categorization
        "priority": incident["severity"], # Ticket priority derived from incident severity
        "affected_services": incident["affected_services"],
        "description": { # Detailed description for the ticket
            "summary": incident["root_cause"],
            "first_seen": incident["first_seen"],
            "last_seen": incident["last_seen"],
            "confidence": incident["confidence"],
            "evidence_samples": incident["evidence_samples"],
            "remediation": incident["remediation"],
            "validation_checks": incident["validation_checks"]
        }
    }

# Iterate through the top 3 incidents (sorted by severity and confidence) to generate and print previews.
for _, inc in top_incidents.head(3).iterrows():
    incident = inc.to_dict() # Convert the incident Series to a dictionary for easier access.
    print("\n" + "="*100) # Print a separator for readability.
    # Print the generated Slack message preview.
    print(slack_message_preview(incident))
    print("Ticket payload:")
    # Print the JSON representation of the ticket payload, truncated for display.
    print(json.dumps(ticket_payload_preview(incident), indent=2)[:3000])


🚨 *Incident Detected: Memory Cpu*

*Severity:* P1
*Confidence:* 0.94
*Affected services:* kubelet, market-data-consumer, pricing-service, risk-engine
*Evidence count:* 355
*Likely cause:* Likely resource pressure from CPU throttling, memory leak, OOM restarts, or JVM GC pauses.
*Top evidence:* `container_cpu_cfs_throttled_periods_total increased sharply sample=437b29 latency_ms=62564`
*Recommended first action:* Inspect CPU/memory usage and container limits.

_Status: Preview only. Human approval required before sending._

Ticket payload:
{
  "title": "P1: Memory Cpu incident detected",
  "labels": [
    "ai-generated",
    "incident",
    "P1",
    "memory_cpu"
  ],
  "priority": "P1",
  "affected_services": [
    "kubelet",
    "market-data-consumer",
    "pricing-service",
    "risk-engine"
  ],
  "description": {
    "summary": "Likely resource pressure from CPU throttling, memory leak, OOM restarts, or JVM GC pauses.",
    "first_seen": "2026-05-09T15:12:26.000Z",
    "last_seen"

## 9. Cookbook Synthesizer Agent

This creates practical checklists your team can reuse as runbooks.

In [26]:
# ============================================================
# 12. Cookbook Synthesizer Agent
# ============================================================
# This agent generates incident response 'cookbooks' or runbooks
# based on the identified incidents. These cookbooks provide step-by-step
# guidance for incident responders, making the resolution process faster and more consistent.

def cookbook_for_incident(incident):
    """Generates a markdown-formatted cookbook for a given incident."""
    # Construct the title of the runbook, including the incident category and severity.
    title = f"Runbook: {incident['category'].replace('_',' ').title()} - {incident['severity']}"
    # Initialize a list to hold all lines of the markdown document.
    lines = [f"# {title}", ""]

    lines.append("## When to use")
    # Provide guidance on when to use this specific runbook, based on confidence score.
    lines.append(f"- Use when logs show signals similar to this incident category with confidence >= {incident['confidence']}.")
    lines.append("")

    lines.append("## Evidence to confirm")
    # List the top 5 evidence samples to help responders quickly confirm the incident.
    for ev in incident["evidence_samples"][:5]:
        lines.append(f"- `{ev.get('message','')[:220]}`") # Truncate long messages for readability
    lines.append("")

    lines.append("## Immediate actions")
    # Detail the recommended remediation steps as ordered actions.
    for i, step in enumerate(incident["remediation"], start=1):
        lines.append(f"{i}. {step}")
    lines.append("")

    lines.append("## Validation checks")
    # List the checks to perform after remediation to validate the fix.
    for i, check in enumerate(incident["validation_checks"], start=1):
        lines.append(f"{i}. {check}")
    lines.append("")

    lines.append("## Safety")
    # Include important safety notes for incident responders.
    lines.append("- Use human approval before customer-impacting actions.")
    lines.append("- Prefer rollback or containment before risky live changes.")

    return "\n".join(lines)

# Initialize an empty list to store the markdown text for each cookbook.
cookbook_text = []
# Iterate through all identified incidents.
for inc in incidents:
    # Only generate cookbooks for P1 and P2 severity incidents.
    if inc["severity"] in ["P1", "P2"]:
        cookbook_text.append(cookbook_for_incident(inc))

# Define the output path for the generated cookbook file.
cookbook_path = OUTPUT_DIR / "incident_cookbook.md"
# Write all generated cookbook text to a single markdown file, separated by horizontal rules.
cookbook_path.write_text("\n\n---\n\n".join(cookbook_text), encoding="utf-8")

# Print the path where the cookbook was saved.
print("Cookbook written to:", cookbook_path)
# Display a preview of the generated cookbook by printing its first 2500 characters.
print(cookbook_path.read_text(encoding="utf-8")[:2500])

Cookbook written to: /content/devops_incident_suite/outputs/incident_cookbook.md
# Runbook: Memory Cpu - P1

## When to use
- Use when logs show signals similar to this incident category with confidence >= 0.94.

## Evidence to confirm
- `container_cpu_cfs_throttled_periods_total increased sharply sample=437b29 latency_ms=62564`
- `OldGen usage above threshold`
- `Container market-data-consumer was OOMKilled sample=ed3be1 latency_ms=53169`
- `container_cpu_cfs_throttled_periods_total increased sharply sample=5070c4 latency_ms=33334`
- `heap usage reached 97 percent before restart sample=55e485 latency_ms=67639`

## Immediate actions
1. Inspect CPU/memory usage and container limits.
2. Review heap/GC metrics and recent traffic changes.
3. Scale replicas or temporarily raise resource limits if safe.
4. Capture heap/thread dump for suspected leak.
5. Rollback recent release if resource spike started after deploy.

## Validation checks
1. Pod restarts stop.
2. CPU throttling and GC pause m

## 10. Final Incident Report

This generates the main artifact you can show in your hackathon demo.

In [32]:
# ============================================================
# 13. Final Report Builder
# ============================================================
# This agent is responsible for synthesizing all the identified incidents
# and their details into a comprehensive markdown report. It also saves
# the incident data in JSON and CSV formats for further analysis or integration.

def build_markdown_report(incidents):
    """Constructs a comprehensive markdown report detailing all detected incidents."""
    lines = []
    lines.append("# Multi-Agent DevOps Incident Analysis Report")
    lines.append("")
    # Add a timestamp for when the report was generated.
    lines.append(f"Generated at: {datetime.now(timezone.utc).isoformat()}")
    lines.append("")
    lines.append("## Executive Summary")
    lines.append("")
    # Calculate summary statistics for incident severities and categories.
    sev_counts = Counter([i["severity"] for i in incidents])
    cat_counts = Counter([i["category"] for i in incidents])
    lines.append(f"- Total incident candidates: {len(incidents)}")
    lines.append(f"- Severity counts: {dict(sev_counts)}")
    lines.append(f"- Categories detected: {dict(cat_counts)}")
    lines.append("")
    lines.append("## Incident Candidates")
    lines.append("")
    # Iterate through each incident, sorting by severity and then confidence,
    # to present them in a structured manner.
    for idx, inc in enumerate(sorted(incidents, key=lambda x: (x["severity"], -x["confidence"])), start=1):
        lines.append(f"### {idx}. {inc['category'].replace('_',' ').title()} — {inc['severity']}")
        lines.append("")
        lines.append(f"- Confidence: `{inc['confidence']}`")
        lines.append(f"- Evidence count: `{inc['evidence_count']}`")
        lines.append(f"- Affected services: `{', '.join(inc['affected_services'])}`")
        lines.append(f"- First seen: `{inc['first_seen']}`")
        lines.append(f"- Last seen: `{inc['last_seen']}`")
        lines.append(f"- Ticket required: `{inc['ticket_required']}`")
        lines.append(f"- Notification required: `{inc['notification_required']}`")
        lines.append("")
        lines.append("#### Likely Root Cause")
        lines.append(inc["root_cause"])
        lines.append("")
        lines.append("#### Evidence Samples")
        # Display a limited number of evidence samples for conciseness.
        for ev in inc["evidence_samples"][:5]:
            lines.append(f"- `{ev.get('source_file')}:{ev.get('line_no')}` [{ev.get('level')}] `{ev.get('service')}` — {ev.get('message')}")
        lines.append("")
        lines.append("#### Remediation")
        for step in inc["remediation"]:
            lines.append(f"- {step}")
        lines.append("")
        lines.append("#### Validation")
        for check in inc["validation_checks"]:
            lines.append(f"- {check}")
        lines.append("")
    return "\n".join(lines)

# Generate the markdown report using the `build_markdown_report` function.
report = build_markdown_report(incidents)

# Define output paths for the generated report and data files.
report_path = OUTPUT_DIR / "incident_report.md"
json_path = OUTPUT_DIR / "incident_results.json"
csv_path = OUTPUT_DIR / "incident_results.csv"

# Write the markdown report to a file.
report_path.write_text(report, encoding="utf-8")
# Write the incident data as a pretty-printed JSON file.
json_path.write_text(json.dumps(incidents, indent=2), encoding="utf-8")
# Write the incident data to a CSV file.
incidents_df.to_csv(csv_path, index=False)

# Print the paths of the generated output files.
print("Wrote:")
print("-", report_path)
print("-", json_path)
print("-", csv_path)
print("\nPreview:")
# Display a preview of the markdown report (first 3000 characters).
print(report[:3000])

Wrote:
- /content/devops_incident_suite/outputs/incident_report.md
- /content/devops_incident_suite/outputs/incident_results.json
- /content/devops_incident_suite/outputs/incident_results.csv

Preview:
# Multi-Agent DevOps Incident Analysis Report

Generated at: 2026-05-08T21:00:13.580169+00:00

## Executive Summary

- Total incident candidates: 74
- Severity counts: {'P1': 9, 'P2': 64, 'P3': 1}
- Categories detected: {'memory_cpu': 2, 'database': 4, 'deployment_regression': 14, 'disk_storage': 2, 'external_dependency': 14, 'queue_backlog': 4, 'network': 2, 'authentication': 14, 'api_timeout': 14, 'unknown': 4}

## Incident Candidates

### 1. Memory Cpu — P1

- Confidence: `0.94`
- Evidence count: `355`
- Affected services: `kubelet, market-data-consumer, pricing-service, risk-engine`
- First seen: `2026-05-09T15:12:26.000Z`
- Last seen: `2026-05-09T16:17:36.000Z`
- Ticket required: `True`
- Notification required: `True`

#### Likely Root Cause
Likely resource pressure from CPU throttl

In [19]:
# ============================================================
# 14. Download generated outputs
# ============================================================
from google.colab import files

zip_out = OUTPUT_DIR / "incident_analysis_outputs.zip"
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.glob("*"):
        if p.is_file() and p.name != zip_out.name:
            z.write(p, p.name)

print("Output package:", zip_out)
files.download(str(zip_out))

Output package: /content/devops_incident_suite/outputs/incident_analysis_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 11. Optional Eval/Scoring

If your uploaded ZIP contains `ground_truth_eval_only/golden_labels.json`, this section compares inferred categories/severities against hidden truth.

Important:
- Do **not** feed ground truth into agents.
- Use it only after inference.

In [34]:
# ============================================================
# 15. Locate ground truth files, if present
# ============================================================
# This cell is responsible for locating ground truth JSON files within the
# extracted log directories. These files contain 'golden labels' that are used
# for evaluating the performance of the incident detection and classification
# pipeline against known correct answers.

# `gt_files = list(EXTRACT_DIR.rglob("ground_truth_eval_only/golden_labels.json"))`:
#   - `EXTRACT_DIR`: This is the base directory where all uploaded ZIP files
#     have been extracted and individual log files are located.
#   - `.rglob("ground_truth_eval_only/golden_labels.json")`: This method
#     recursively searches for all files named 'golden_labels.json' within
#     any subdirectory named 'ground_truth_eval_only' starting from `EXTRACT_DIR`.
#     This allows the system to find ground truth files even if they are deeply
#     nested within the extracted log corpus structure.
#   - `list(...)`: Converts the generator returned by `rglob` into a list
#     of Path objects, each pointing to a discovered ground truth file.
gt_files = list(EXTRACT_DIR.rglob("ground_truth_eval_only/golden_labels.json"))

# `if not gt_files:`:
#   - Checks if the `gt_files` list is empty. If no 'golden_labels.json' files
#     were found, it means there's no ground truth data available for evaluation.
#   - In this case, a message is printed indicating that the evaluation step
#     will be skipped.
if not gt_files:
    print("No ground_truth_eval_only/golden_labels.json found. Skipping eval.")
# `else:`:
#   - If ground truth files were found, this block is executed.
#   - A message is printed to inform the user that ground truth files have been
#     located.
#   - A loop iterates through each discovered ground truth file and prints its
#     path, providing transparency about which files will be used for evaluation.
else:
    print("Found ground truth files:")
    for p in gt_files:
        print("- ", p)

Found ground truth files:
-  /content/devops_incident_suite/extracted/devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block2_full_raw/ground_truth_eval_only/golden_labels.json
-  /content/devops_incident_suite/extracted/devops_raw_log_corpus_v2_all/devops_raw_log_corpus_v2/block1_quick_raw/ground_truth_eval_only/golden_labels.json


## Summary of Key Performance Metrics from `incident_results.csv`

In [33]:
# Total number of incidents
total_incidents = len(incidents_df)
print(f"Total Number of Incidents Detected: {total_incidents}")

print("\n--- Incidents by Category ---")
print(incidents_df['category'].value_counts())

print("\n--- Incidents by Severity ---")
print(incidents_df['severity'].value_counts())

print("\n--- Confidence Score ---")
print(f"Average Confidence Score: {incidents_df['confidence'].mean():.2f}")
print(f"Standard Deviation of Confidence Score: {incidents_df['confidence'].std():.2f}")

print("\n--- Most Affected Services ---")
# Flatten the list of affected services and count occurrences
all_affected_services = [service for sublist in incidents_df['affected_services'] for service in sublist]
service_counts = Counter(all_affected_services)
# Display the top 10 most affected services
for service, count in service_counts.most_common(10):
    print(f"- {service}: {count} incidents")

Total Number of Incidents Detected: 74

--- Incidents by Category ---
category
external_dependency      14
deployment_regression    14
api_timeout              14
authentication           14
unknown                   4
database                  4
queue_backlog             4
memory_cpu                2
disk_storage              2
network                   2
Name: count, dtype: int64

--- Incidents by Severity ---
severity
P2    64
P1     9
P3     1
Name: count, dtype: int64

--- Confidence Score ---
Average Confidence Score: 0.64
Standard Deviation of Confidence Score: 0.12

--- Most Affected Services ---
- order-api: 59 incidents
- payment-api: 48 incidents
- market-data-consumer: 45 incidents
- risk-engine: 45 incidents
- nginx-gateway: 45 incidents
- notification-worker: 45 incidents
- postgres-primary: 44 incidents
- external-monitor: 44 incidents
- kafka-broker: 44 incidents
- kubelet: 42 incidents


In [35]:
# ============================================================
# 16. Basic eval scoring
# ============================================================
# This cell performs a basic evaluation of the incident detection pipeline
# by comparing its predictions against ground truth labels, if available.
# It calculates 'category recall' and 'severity accuracy' to assess performance.

def normalize_cat(x):
    """Normalizes a category string by lowercasing, replacing spaces/slashes with underscores, and stripping whitespace."""
    # Converts category names to a consistent format for comparison (e.g., "API Timeout" -> "api_timeout").
    return str(x).lower().replace("/", "_").replace(" ", "_").strip()

def load_ground_truth(gt_files):
    """Loads ground truth incident data from a list of JSON files."""
    records = []
    # Iterate through each ground truth file path.
    for gt in gt_files:
        # Read and parse the JSON content of the ground truth file.
        data = json.loads(gt.read_text(encoding="utf-8"))
        # Iterate through each item (incident) in the parsed JSON data.
        for item in data:
            records.append({
                "case_id": item.get("case_id"), # Unique identifier for the ground truth case.
                "raw_file": item.get("raw_file"), # The raw log file associated with this ground truth.
                "expected_category": normalize_cat(item.get("expected_category")), # Expected incident category.
                "expected_severity": item.get("expected_severity"), # Expected incident severity.
                "affected_services": item.get("affected_services", []), # List of affected services.
            })
    # Return a Pandas DataFrame containing all ground truth records.
    return pd.DataFrame(records)

# Check if any ground truth files were found in the previous step.
if gt_files:
    # Load ground truth data into a DataFrame.
    gt_df = load_ground_truth(gt_files)
    # Create a copy of the incident predictions DataFrame for manipulation.
    pred_df = incidents_df.copy()
    # Add a normalized source file name column to the predictions for matching.
    pred_df["source_file_norm"] = pred_df["incident_id"].apply(lambda x: x.split("::")[0])
    # Add a normalized category column to the predictions for matching.
    pred_df["category_norm"] = pred_df["category"].apply(normalize_cat)

    scored = []
    # Iterate through each ground truth incident to score predictions.
    for _, gt in gt_df.iterrows():
        # Find prediction candidates that match the ground truth's raw file and category.
        candidates = pred_df[
            pred_df["source_file_norm"].str.endswith(str(gt["raw_file"]).split("/")[-1]) &
            (pred_df["category_norm"] == gt["expected_category"])
        ]
        # Determine if the category was detected by the pipeline.
        category_detected = len(candidates) > 0
        severity_match = False
        best_conf = None
        pred_sev = None
        # If the category was detected, proceed to check severity and confidence.
        if category_detected:
            # Get the best candidate (highest confidence) for this ground truth.
            best = candidates.sort_values("confidence", ascending=False).iloc[0]
            pred_sev = best["severity"]
            best_conf = best["confidence"]
            # Check if the predicted severity matches the expected severity.
            severity_match = pred_sev == gt["expected_severity"]
        # Append the scoring results for the current ground truth incident.
        scored.append({
            "case_id": gt["case_id"],
            "raw_file": gt["raw_file"],
            "expected_category": gt["expected_category"],
            "expected_severity": gt["expected_severity"],
            "category_detected": category_detected,
            "predicted_severity": pred_sev,
            "severity_match": severity_match,
            "confidence": best_conf
        })

    # Convert the scored results into a DataFrame for analysis.
    score_df = pd.DataFrame(scored)
    display(score_df)

    # Calculate category recall: the proportion of ground truth categories that were detected.
    category_recall = score_df["category_detected"].mean()
    # Calculate severity accuracy: the proportion of correctly predicted severities,
    # only considering incidents where the category was detected.
    severity_accuracy_on_detected = score_df[score_df["category_detected"]]["severity_match"].mean()

    # Print the calculated evaluation metrics.
    print(f"Category recall: {category_recall:.2%}")
    print(f"Severity accuracy on detected categories: {severity_accuracy_on_detected:.2%}")

    # Define the output path for the evaluation scores CSV.
    score_path = OUTPUT_DIR / "eval_scores.csv"
    # Write the evaluation scores to a CSV file.
    score_df.to_csv(score_path, index=False)
    print("Eval scores written to:", score_path)


,case_id,raw_file,expected_category,expected_severity,category_detected,predicted_severity,severity_match,confidence
0,DB_POOL,raw_logs/full_mixed_window_01.jsonl,database,P1,True,P1,True,0.90
1,DB_DEADLOCK,raw_logs/full_mixed_window_01.jsonl,database,P2,True,P1,False,0.90
2,DB_SLOW_QUERY,raw_logs/full_mixed_window_01.jsonl,database,P2,True,P1,False,0.90
3,NET_RESET,raw_logs/full_mixed_window_02.jsonl,network,P2,True,P2,True,0.93
4,NET_DNS,raw_logs/full_mixed_window_02.jsonl,network,P2,True,P2,True,0.93
5,NET_TLS,raw_logs/full_mixed_window_02.jsonl,network,P2,True,P2,True,0.93
6,AUTH_JWT,raw_logs/full_mixed_window_03.jsonl,authentication,P2,True,P2,True,0.88
7,AUTH_JWK,raw_logs/full_mixed_window_03.jsonl,authentication,P2,True,P2,True,0.88
8,AUTH_RBAC,raw_logs/full_mixed_window_03.jsonl,authentication,P3,True,P2,False,0.88
9,CPU_THROTTLE,raw_logs/full_mixed_window_04.jsonl,memory_cpu,P2,True,P1,False,0.94


Category recall: 100.00%
Severity accuracy on detected categories: 61.90%
Eval scores written to: /content/devops_incident_suite/outputs/eval_scores.csv


# 12. How to Upgrade This into a Full LangGraph App

Use this notebook as the reference implementation.

## Recommended LangGraph nodes

```text
START
 -> ingest_logs
 -> parse_logs
 -> extract_signals
 -> correlate_windows
 -> classify_incidents
 -> build_timeline
 -> root_cause_analysis
 -> remediation_mapping
 -> human_approval_gate
 -> notification_preview
 -> ticket_preview
 -> cookbook_generation
 -> report_builder
END
```

## State object

```python
class IncidentState(TypedDict):
    raw_files: list[str]
    parsed_logs: list[ParsedLog]
    evidence: list[dict]
    incident_candidates: list[IncidentCandidate]
    approved_actions: dict
    final_report: str
```

## Why LangGraph shines here

Use conditional routing:

```text
If severity in P1/P2:
    route to notification + ticket agents
Else:
    route only to report/cookbook
```

Use parallelism:

```text
Parser Agent
   ↓
Signal Extractor
   ↓
[Classifier, Timeline, RCA, Severity] in parallel
   ↓
Merge
```

Use human-in-the-loop:

```text
Approve Slack?
Approve ticket?
Approve remediation?
```

# 13. RAG Plan

RAG is where this project becomes production-grade.

## What to index

Create a knowledge base from:

1. Internal runbooks
2. Past incident reports
3. Postmortems
4. Service ownership docs
5. Architecture diagrams
6. Deployment guides
7. Error code catalog
8. Known remediation recipes

## Chunking strategy

Use chunks like:

```text
service_name
incident_type
symptoms
diagnostic_commands
safe_remediation
rollback_plan
validation_steps
owner_team
```

## Retrieval query

For each incident candidate, retrieve using:

```text
category + affected_service + top error messages + suspected root cause
```

Example:

```text
payment-api HikariPool connection unavailable DB_POOL_TIMEOUT remediation runbook
```

## RAG output

The RCA agent should receive:

```text
raw evidence
+
retrieved runbook passages
+
past incident examples
```

Then produce grounded recommendations.

## Minimal implementation

Use:

```text
FAISS + sentence-transformers
```

or hosted:

```text
Pinecone / Weaviate / Azure AI Search / OpenSearch
```

# 14. Scoring and Evals Roadmap

## Eval levels

### Level 1 — Detection
Did the system detect the right category?

Metric:
```text
category recall
```

### Level 2 — Severity
Did it classify P1/P2/P3 correctly?

Metric:
```text
severity accuracy
```

### Level 3 — Evidence grounding
Does each incident cite actual log lines?

Metric:
```text
percentage of RCA claims with evidence references
```

### Level 4 — Remediation quality
Does the fix match the incident?

Metric:
```text
keyword overlap + human score
```

### Level 5 — Action correctness
Were notification/ticket actions triggered only when needed?

Metric:
```text
precision and recall for ticket_required / notification_required
```

## Suggested scorecard

| Metric | Target for demo |
|---|---:|
| Category recall | > 85% |
| Severity accuracy | > 75% |
| Evidence grounding | > 90% |
| Ticket trigger accuracy | > 85% |
| Hallucination rate | < 5% |

## Human eval rubric

Ask reviewers to rate each RCA from 1-5 on:

1. Correct category
2. Correct severity
3. Evidence quality
4. Remediation usefulness
5. Safety
6. Clarity

# 15. Next Production Steps

## Add LLM RCA

Use the deterministic pipeline for evidence and classification. Then ask the LLM to write:

- RCA summary
- stakeholder-friendly summary
- remediation rationale
- post-incident learnings

Keep evidence lines in the prompt to reduce hallucination.

## Add real integrations

Start with preview mode, then add adapters:

```text
NotificationAdapter:
- Slack
- Teams
- Discord
- Email
- Preview

TicketAdapter:
- JIRA
- GitHub Issues
- Trello
- ServiceNow
- Local JSON
```

## Add observability

Track:

```text
input token count
output token count
agent latency
retrieval hit rate
eval score
human override rate
```

## Add governance

Require human approval before:

```text
sending notification
creating ticket
executing remediation
marking incident resolved
```